# 1. Preprocessing

In [17]:
import pandas as pd
import numpy as np 
import sklearn
import networkx as nx
import ast

import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


In [ ]:
train = pd.read_csv("./data/train.csv")
test = pd.read_csv("./data/test.csv")

In [ ]:
print(train.head())

In [ ]:
print(test.head())

## Binary Classification Setup 

### Feature Engineering

In [21]:
def get_graph_features(G, node):
    """Extract rich graph features for a given node in tree G."""
    features = {}

    # Degree-based (4 features)
    # features['in_degree'] = G.in_degree(node)
    # features['out_degree'] = G.out_degree(node)
    # features['degree'] = G.degree(node)
    features['is_leaf'] = int(G.degree(node) == 1)

    # Subtree size (1 features)
    # features['descendants'] = len(nx.descendants(G, node))

    # Centralities (5 features)
    closeness = nx.closeness_centrality(G)
    betweenness = nx.betweenness_centrality(G)
    pagerank = nx.pagerank(G, alpha=0.85)
    harmonic = nx.harmonic_centrality(G)
    degree = nx.degree_centrality(G)
    try:
        eigen = nx.eigenvector_centrality_numpy(G)
    except nx.NetworkXException:
        eigen = {n: 0.0 for n in G.nodes()}

    features['closeness'] = closeness.get(node, 0)
    features['betweenness'] = betweenness.get(node, 0)
    features['pagerank'] = pagerank.get(node, 0)
    features['harmonic'] = harmonic.get(node, 0)
    features['degree_cnt'] = degree.get(node, 0)

    return features


### Graph Flatten and Processing

In [ ]:
def flatten_and_add_graph_features(df, is_test: bool) -> pd.DataFrame:
    """
    Processes a dependency-tree dataframe into node-level format with graph features.
    Works on both train and test DataFrames.
    """
    rows = []

    for idx, row in df.iterrows():
        lang = row['language']
        sent_id = row['sentence']
        n = row['n']
        edges_raw = row['edgelist']
        is_root = row['root']


        # Convert string to list of tuples
        if isinstance(edges_raw, str):
            edges = ast.literal_eval(edges_raw)
        else:
            edges = edges_raw  
        G = nx.DiGraph(edges)

        for node in G.nodes():
            node_data = {
                'language': lang,
                'sentence_id': sent_id,
                'n': n,
                'vertex_id': node
            }
            if not is_test:
                node_data['root'] = is_root
                
            node_data.update(get_graph_features(G, node))
            rows.append(node_data)

    return pd.DataFrame(rows)


binary_train = flatten_and_add_graph_features(train, is_test=False)
binary_test = flatten_and_add_graph_features(test, is_test=True)

In [ ]:
binary_train.head()

In [ ]:
binary_test.head()

### Load Processed Data

In [ ]:
binary_train.to_csv("./data/processed_binary_train.csv")
binary_test.to_csv("./data/processed_binary_test.csv")

## Graph Classification Setup

### Extract and Process data

In [26]:
def sentence_to_graph(row, is_test=False):
    
    edges = ast.literal_eval(row['edgelist']) if isinstance(row['edgelist'], str) else row['edgelist']
    G = nx.DiGraph(edges)
    nodes = sorted(G.nodes())
    node_id_map = {node_id: idx for idx, node_id in enumerate(nodes)}

    edge_index = torch.tensor(
        [[node_id_map[src], node_id_map[dst]] for src, dst in edges],
        dtype=torch.long
    ).t().contiguous()

    # Feature Engineering (degree, etc.)
    degrees = [G.degree[node] for node in nodes]
    x = torch.tensor([[deg] for deg in degrees], dtype=torch.float)

    # Labeling the root node for training (1 if root, 0 otherwise)
    if not is_test:
        y = torch.tensor(
            [1 if node == row['root'] else 0 for node in nodes],  # root node gets label 1, others get 0
            dtype=torch.long
        )
    else:
        # During inference, no root information is passed, just placeholders for labels
        y = torch.zeros(len(nodes), dtype=torch.long)

#     data = Data(
#         x=x,
#         edge_index=edge_index,
#         y=y
#     )
#     data.num_nodes = len(nodes)
#     data.language = row['language']
#     data.sentence_id = row['sentence']

    # Optional: keep mapping to original node ids if needed
    data.node_ids = torch.tensor(nodes, dtype=torch.long)
    return data


In [ ]:
train_graph = [sentence_to_graph(row, is_test=False) for _, row in train.iterrows()]
test_graph = [sentence_to_graph(row, is_test=True) for _, row in test.iterrows()]

### Load 

In [ ]:
torch.save(train_graph, './data/processed_graph_train.pt')
torch.save(test_graph, './data/processed_graph_test.pt')